# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [14]:
!git clone https://github.com/flyrank-bih/flyrank-ml-internship-starter.git

import os
os.chdir("flyrank-ml-internship-starter")

print(os.getcwd())

Cloning into 'flyrank-ml-internship-starter'...
remote: Enumerating objects: 283, done.
remote: Counting objects: 100% (136/136), done.
remote: Compressing objects: 100% (58/58), done.
remote: Total 283 (delta 106), reused 78 (delta 78), pack-reused 147 (from 1)
Receiving objects: 100% (283/283), 1.85 MiB | 11.58 MiB/s, done.
Resolving deltas: 100% (152/152), done.
/content/flyrank-ml-internship-starter


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [2]:
''' One row represents one webpage's search performance for a specific day.
 Each row contains search metrics that can be used to determine whether the page should be prioritized for content refresh.'''

" One row represents one webpage's search performance for a specific day.\n Each row contains search metrics that can be used to determine whether the page should be prioritized for content refresh."

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [4]:
'''The features used for this project include impressions, clicks, CTR,
average position, and content age, as they are available before making a refresh decision.
The label is the declining page indicator (or refresh target), which identifies pages requiring review.
Context fields such as content ID, client ID, and date provide identification and grouping but are not used directly for prediction.
Label-derived fields such as trend_direction and trend_pct are deliberately excluded because they reveal the target and would introduce data leakage,
resulting in unrealistic model performance.'''

'The features used for this project include impressions, clicks, CTR,\naverage position, and content age, as they are available before making a refresh decision.\nThe label is the declining page indicator (or refresh target), which identifies pages requiring review.\nContext fields such as content ID, client ID, and date provide identification and grouping but are not used directly for prediction.\nLabel-derived fields such as trend_direction and trend_pct are deliberately excluded because they reveal the target and would introduce data leakage,\nresulting in unrealistic model performance.'

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [5]:
'''To verify the data contract, I run queries that confirm the grain of the dataset by showing that each row represents one webpage on one day.
I then calculate the total number of rows and the minimum and maximum dates for the selected month to verify the time window.
Finally, I check the availability and completeness of the selected features by counting non-missing values
and filtering valid records using the appropriate availability condition.
These queries ensure that every assumption about the dataset is supported by actual data rather than guesswork.'''

'To verify the data contract, I run queries that confirm the grain of the dataset by showing that each row represents one webpage on one day.\nI then calculate the total number of rows and the minimum and maximum dates for the selected month to verify the time window.\nFinally, I check the availability and completeness of the selected features by counting non-missing values\nand filtering valid records using the appropriate availability condition.\nThese queries ensure that every assumption about the dataset is supported by actual data rather than guesswork.'

In [11]:
  '''SELECT content_id,date,impressions,clicks,ctr,avg_position
  FROM fact_content_daily_performance
WHERE month = '2026-03'
LIMIT 10;'''

'''select count(*) as total_rows,
min(data) as start_date,
max(data) as end_date
from fact_content_daily_performance
where month='2026-03'''

'''SELECT
    COUNT(*) AS available_rows
FROM fact_content_daily_performance
WHERE month = '2026-03'
  AND is_available IS TRUE;'''

  '''SELECT
    COUNT(*) AS total_rows,
    COUNT(impressions) AS impressions_present,
    COUNT(clicks) AS clicks_present,
    COUNT(ctr) AS ctr_present,
    COUNT(avg_position) AS avg_position_present,
    COUNT(content_age_days) AS content_age_present
FROM fact_content_daily_performance
WHERE month = '2026-03';'''

"SELECT\n  COUNT(*) AS total_rows,\n  COUNT(impressions) AS impressions_present,\n  COUNT(clicks) AS clicks_present,\n  COUNT(ctr) AS ctr_present,\n  COUNT(avg_position) AS avg_position_present,\n  COUNT(content_age_days) AS content_age_present\nFROM fact_content_daily_performance\nWHERE month = '2026-03';"

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [6]:
''' This dataset supports decision-making but cannot prove cause-and-effect relationships or explain Google's ranking algorithm.
It may not capture all external factors affecting search performance, such as algorithm updates or user behavior outside the available data.
Therefore, the model provides evidence-based recommendations rather than guaranteed outcomes,
and its predictions should be interpreted as decision support rather than absolute truth.'''

" This dataset supports decision-making but cannot prove cause-and-effect relationships or explain Google's ranking algorithm.\nIt may not capture all external factors affecting search performance, such as algorithm updates or user behavior outside the available data.\nTherefore, the model provides evidence-based recommendations rather than guaranteed outcomes,\nand its predictions should be interpreted as decision support rather than absolute truth."

In [16]:
import pandas as pd

# Load the processed dataset
df = pd.read_csv("/content/flyrank-ml-internship-starter/data/raw/content_refresh_anonymized.csv")

# Select only the required columns
feature_df = df[[
    "content_id",
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "content_age_days"
]]

# Display first 10 rows
feature_df.head(10)

,content_id,impressions_90d,clicks_90d,ctr,avg_position,content_age_days
0,content_304f48230142,3803,29,0.76,10.6,187
1,content_a1fb4e703a9e,15320,7,0.05,20.3,445
2,content_9aa793d4d895,12581,11,0.09,36.5,141
3,content_331d6c4de07b,11751,58,0.49,6.2,463
4,content_d99b7a2d90ca,19140,24,0.13,44.0,263
5,content_d4084a4bc775,3970,1,0.03,8.5,147
6,content_9a34b442b552,20,0,0.00,7.0,90
7,content_a63219c6e95a,1724,1,0.06,21.2,445
8,content_5e6c160719bc,32574,29,0.09,46.0,90
9,content_c27558df2b0c,1240,2,0.16,4.9,257


In [17]:
# Honest feature set
honest_features = [
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "content_age_days"
]

print("Honest Features:")
print(honest_features)

# Add a leaking feature
leaky_features = honest_features + ["trend_direction"]

print("\nLeaky Features:")
print(leaky_features)

print("\nObservation:")
print("Including 'trend_direction' would produce an unrealistically high model score because it directly reveals the target label. Therefore, it must be removed.")

Honest Features:
['impressions_90d', 'clicks_90d', 'ctr', 'avg_position', 'content_age_days']

Leaky Features:
['impressions_90d', 'clicks_90d', 'ctr', 'avg_position', 'content_age_days', 'trend_direction']

Observation:
Including 'trend_direction' would produce an unrealistically high model score because it directly reveals the target label. Therefore, it must be removed.


Features

In [12]:
'''Feature 1: Impressions (impressions_90d)
Why is it available before prediction?
Impressions are recorded from historical Search Console data before the content refresh decision is made, so they are already known at prediction time.

Feature 2: Clicks (clicks_90d)
Why is it available before prediction?
Clicks are historical user interactions collected before making the refresh recommendation, making them available as an input feature.

Feature 3: Click-Through Rate (CTR)
Why is it available before prediction?
CTR is calculated from historical clicks and impressions, both of which are known before deciding whether a page should be refreshed.

Feature 4: Average Position (avg_position)
Why is it available before prediction?
Average search position is historical ranking information already available from previous search performance reports.

Feature 5: Content Age (content_age_days)
Why is it available before prediction?
Content age is calculated from the publication date of the webpage, which is known before any prediction is made.'''

'Feature 1: Impressions (impressions_90d)\nWhy is it available before prediction?\nImpressions are recorded from historical Search Console data before the content refresh decision is made, so they are already known at prediction time.\n\nFeature 2: Clicks (clicks_90d)\nWhy is it available before prediction?\nClicks are historical user interactions collected before making the refresh recommendation, making them available as an input feature.\n\nFeature 3: Click-Through Rate (CTR)\nWhy is it available before prediction?\nCTR is calculated from historical clicks and impressions, both of which are known before deciding whether a page should be refreshed.\n\nFeature 4: Average Position (avg_position)\nWhy is it available before prediction?\nAverage search position is historical ranking information already available from previous search performance reports.\n\nFeature 5: Content Age (content_age_days)\nWhy is it available before prediction?\nContent age is calculated from the publication dat

Data Limitations

In [13]:
## Data Limitation

'''This analysis uses historical search performance data and therefore cannot prove why a webpage gains or loses rankings.
External factors such as Google's algorithm updates, competitor changes, or user behavior are not fully captured in the dataset.
As a result, the model provides decision-support recommendations rather than guaranteed predictions or causal explanations.'''

"This analysis uses historical search performance data and therefore cannot prove why a webpage gains or loses rankings. \nExternal factors such as Google's algorithm updates, competitor changes, or user behavior are not fully captured in the dataset.\nAs a result, the model provides decision-support recommendations rather than guaranteed predictions or causal explanations."

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.